In [ ]:
import torch
import os
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("❌ GPU NOT AVAILABLE")

In [ ]:
!pip install -q -U \
    diffusers \
    transformers \
    accelerate \
    safetensors \
    sentencepiece \
    protobuf \
    mediapipe \
    opencv-python-headless \
    scipy \
    matplotlib \
    pillow

In [ ]:
import diffusers
import transformers
import accelerate
import mediapipe
import cv2
import scipy
import PIL

print("Diffusers   :", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate  :", accelerate.__version__)
print("MediaPipe   :", mediapipe.__version__)
print("OpenCV      :", cv2.__version__)
print("SciPy       :", scipy.__version__)
print("Pillow      :", PIL.__version__)

In [3]:
from pathlib import Path

BASE = Path("/kaggle/working/rhinoplasty_dataset")

directories = [
    BASE / "patients",
    BASE / "models",
    BASE / "validation",
    BASE / "logs",
    BASE / "batches",
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE)
print("\\nCreated:")
for directory in directories:
    print("✓", directory)


Project directory: /kaggle/working/rhinoplasty_dataset
\nCreated:
✓ /kaggle/working/rhinoplasty_dataset/patients
✓ /kaggle/working/rhinoplasty_dataset/models
✓ /kaggle/working/rhinoplasty_dataset/validation
✓ /kaggle/working/rhinoplasty_dataset/logs
✓ /kaggle/working/rhinoplasty_dataset/batches


In [4]:
import json
from pathlib import Path

# Optional input contract:
# - In a future Kaggle integration, attach a Kaggle Dataset containing manifest.json.
# - This cell searches Kaggle input mounts for that file.
# - If none is found, the notebook remains usable in manual mode.
BASE = Path("/kaggle/working/rhinoplasty_dataset")
MANIFEST_SEARCH_ROOT = Path("/kaggle/input")
manifest_candidates = (
    sorted(MANIFEST_SEARCH_ROOT.glob("**/manifest.json"))
    if MANIFEST_SEARCH_ROOT.exists()
    else []
)

MANIFEST_PATH = manifest_candidates[0] if manifest_candidates else None

if MANIFEST_PATH:
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        GENERATION_MANIFEST = json.load(f)
    print("Manifest loaded:", MANIFEST_PATH)
    print("Batch ID:", GENERATION_MANIFEST.get("batch_id", "batch-001"))
else:
    GENERATION_MANIFEST = None
    print("No manifest.json found; manual testing mode is active.")

# Manual defaults are used only when no manifest is available.
if GENERATION_MANIFEST is None:
    GENERATION_SETTINGS = {
        "batch_id": "manual-batch",
        "requested_count": 5,
        "prompts": None,  # Uses P01_PROMPT from the generation cell below.
        "seeds": [0, 1, 2, 3, 4],
        "width": 384,
        "height": 384,
        "num_inference_steps": 12,
        "guidance_scale": 1.0,
    }
else:
    GENERATION_SETTINGS = {
        "batch_id": str(GENERATION_MANIFEST.get("batch_id", "batch-001")),
        "requested_count": int(GENERATION_MANIFEST.get("requested_count", 0)),
        "prompts": GENERATION_MANIFEST.get("prompts"),
        "seeds": GENERATION_MANIFEST.get("seeds", []),
        "width": int(GENERATION_MANIFEST.get("width", GENERATION_MANIFEST.get("image_width", 384))),
        "height": int(GENERATION_MANIFEST.get("height", GENERATION_MANIFEST.get("image_height", 384))),
        "num_inference_steps": int(GENERATION_MANIFEST.get("num_inference_steps", 12)),
        "guidance_scale": float(GENERATION_MANIFEST.get("guidance_scale", 1.0)),
    }

if GENERATION_SETTINGS["requested_count"] < 1:
    raise ValueError("Manifest requested_count must be at least 1.")

if len(GENERATION_SETTINGS["seeds"]) != GENERATION_SETTINGS["requested_count"]:
    raise ValueError(
        "Manifest must contain exactly one seed per requested image. "
        f"Requested {GENERATION_SETTINGS['requested_count']}, "
        f"received {len(GENERATION_SETTINGS['seeds'])} seeds."
    )

prompts_value = GENERATION_SETTINGS["prompts"]
if prompts_value is not None and not (
    isinstance(prompts_value, list)
    and len(prompts_value) == GENERATION_SETTINGS["requested_count"]
    and all(isinstance(p, str) and p.strip() for p in prompts_value)
):
    raise ValueError("Manifest prompts must be a list of non-empty strings, one per image.")

batch_id = "".join(
    ch if ch.isalnum() or ch in "-_" else "-"
    for ch in GENERATION_SETTINGS["batch_id"]
).strip("-") or "batch-001"

BATCH_DIR = BASE / "batches" / batch_id
BATCH_IMAGE_DIR = BATCH_DIR / "images"
BATCH_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Batch:", batch_id)
print("Requested images:", GENERATION_SETTINGS["requested_count"])
print("Dimensions:", f'{GENERATION_SETTINGS["width"]}x{GENERATION_SETTINGS["height"]}')
print("Output directory:", BATCH_DIR)


No manifest.json found; manual testing mode is active.
Batch: manual-batch
Requested images: 5
Dimensions: 384x384
Output directory: /kaggle/working/rhinoplasty_dataset/batches/manual-batch


In [1]:
import importlib.util

print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

from diffusers import Flux2KleinPipeline

print("FLUX.2 Klein import successful!")

TorchAO installed: False
FLUX.2 Klein import successful!


In [2]:
import torch
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"

pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

pipe.enable_model_cpu_offload()

print("FLUX.2 Klein 4B loaded successfully!")

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

The config attributes {'force_upcast': True} were passed to AutoencoderKLFlux2, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

FLUX.2 Klein 4B loaded successfully!


In [5]:
import torch
from PIL import Image

prompt = """
Photorealistic portrait photograph of a fictional adult person,
front-facing, looking directly at the camera, neutral expression,
natural realistic facial anatomy, natural skin texture,
simple hairstyle, no glasses, no jewelry,
plain neutral background, soft studio lighting,
single face, centered composition.
"""

print("Starting test generation...")

with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        height=384,
        width=384,
        num_inference_steps=4,
        guidance_scale=1.0,
        generator=torch.Generator(device="cpu").manual_seed(42),
    )

test_image = result.images[0]

test_path = "/kaggle/working/rhinoplasty_dataset/test_flux.png"
test_image.save(test_path)

print("✓ Generation successful")
print("Saved:", test_path)
print("Image size:", test_image.size)

Starting test generation...


  0%|          | 0/4 [00:00<?, ?it/s]

✓ Generation successful
Saved: /kaggle/working/rhinoplasty_dataset/test_flux.png
Image size: (384, 384)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import torch

BASE = Path("/kaggle/working/rhinoplasty_dataset")
P01_DIR = BASE / "patients" / "P01"
CANDIDATE_DIR = P01_DIR / "normal" / "candidates"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
BATCH_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

P01_PROMPT = """
Photorealistic clinical-style portrait photograph of a fictional
adult East Asian person, approximately 25 to 35 years old,
front-facing straight-on view, looking directly at the camera,
neutral relaxed facial expression.

Natural normal nose with realistic anatomical proportions,
no obvious nasal hump, no obvious nasal deviation,
natural nasal bridge and tip.

Natural realistic skin texture with subtle pores,
realistic facial anatomy, natural eyes and lips,
stable simple hairstyle.

No glasses, no sunglasses, no hat, no jewelry,
no facial accessories, no heavy makeup.

Even neutral studio lighting, soft natural shadows,
plain neutral background, single person,
centered face, realistic photography,
sharp facial details, no artistic stylization.
"""

requested_count = GENERATION_SETTINGS["requested_count"]
seeds = GENERATION_SETTINGS["seeds"]
prompts = GENERATION_SETTINGS["prompts"]
width = GENERATION_SETTINGS["width"]
height = GENERATION_SETTINGS["height"]
num_inference_steps = GENERATION_SETTINGS["num_inference_steps"]
guidance_scale = GENERATION_SETTINGS["guidance_scale"]

batch_records = []
print(f"Generating {requested_count} independent face image(s) for batch {batch_id}...")

for index in range(requested_count):
    seed = int(seeds[index])
    prompt = prompts[index] if prompts is not None else P01_PROMPT
    filename = f"image_{index + 1:04d}.png"
    batch_path = BATCH_IMAGE_DIR / filename

    print(f"Generating image {index + 1}/{requested_count} | seed={seed}")
    record = {
        "index": index + 1,
        "filename": filename,
        "prompt": prompt,
        "seed": seed,
        "width": width,
        "height": height,
        "status": "FAILED",
        "error": None,
    }

    try:
        generator = torch.Generator(device="cpu").manual_seed(seed)
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                height=height,
                width=width,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=generator,
            )

        image = result.images[0]
        image.save(batch_path, format="PNG")

        # Keep the old candidate naming convention available for the later,
        # unchanged multi-view cells. This compatibility copy is not the batch source of truth.
        legacy_path = CANDIDATE_DIR / f"P01_candidate_{seed}.png"
        image.save(legacy_path, format="PNG")

        record.update({
            "status": "GENERATED",
            "path": str(batch_path),
            "legacy_candidate_path": str(legacy_path),
            "actual_width": image.width,
            "actual_height": image.height,
        })
        print("✓ Saved:", batch_path)
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        print(f"✗ Failed image {index + 1}: {record['error']}")

    batch_records.append(record)

generated_count = sum(r["status"] == "GENERATED" for r in batch_records)
failed_count = requested_count - generated_count
metadata_path = BATCH_DIR / "metadata.json"
summary_path = BATCH_DIR / "batch_summary.json"

metadata_document = {
    "batch_id": batch_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "generation_mode": "manifest" if GENERATION_MANIFEST is not None else "manual",
    "requested_count": requested_count,
    "generated_count": generated_count,
    "failed_count": failed_count,
    "images": batch_records,
}
metadata_path.write_text(json.dumps(metadata_document, indent=2), encoding="utf-8")

summary_document = {
    "job_id": int(GENERATION_MANIFEST.get("job_id", 0)) if GENERATION_MANIFEST else 0,
    "batch_id": batch_id,
    "requested_count": requested_count,
    "generated_count": generated_count,
    "successful_count": generated_count,
    "failed_count": failed_count,
    "image_width": width,
    "image_height": height,
    "output_directory": str(BATCH_DIR),
    "metadata_file": str(metadata_path),
    "images": [
        {
            "filename": r["filename"],
            "seed": r["seed"],
            "status": r["status"],
            "path": r.get("path"),
            "error": r.get("error"),
        }
        for r in batch_records
    ],
}
summary_path.write_text(json.dumps(summary_document, indent=2), encoding="utf-8")

print("\\nBatch finished.")
print(f"Requested: {requested_count} | Generated: {generated_count} | Failed: {failed_count}")
print("Metadata:", metadata_path)
print("Summary:", summary_path)

if failed_count:
    print("WARNING: Some images failed; inspect batch_summary.json and metadata.json.")


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

successful_records = [r for r in batch_records if r["status"] == "GENERATED"]
images = [Image.open(r["path"]).convert("RGB") for r in successful_records]

if not images:
    print("No successfully generated images to display.")
else:
    columns = min(5, len(images))
    rows = (len(images) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(3.5 * columns, 4 * rows))
    axes = list(getattr(axes, "flat", [axes]))

    for i, (ax, image, record) in enumerate(zip(axes, images, successful_records)):
        ax.imshow(image)
        ax.set_title(f"Image {record['index']} | seed={record['seed']}")
        ax.axis("off")

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)

print("Tasks available:",
      hasattr(mp, "tasks"))

print("✓ MediaPipe imported successfully")

In [ ]:
from pathlib import Path
import urllib.request

MODEL_DIR = Path("/kaggle/working/rhinoplasty_dataset/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKER_PATH = MODEL_DIR / "face_landmarker.task"

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

if not LANDMARKER_PATH.exists():
    print("Downloading Face Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, LANDMARKER_PATH)
    print("✓ Download complete")
else:
    print("✓ Model already exists")

print("Model:", LANDMARKER_PATH)
print("Size:", round(LANDMARKER_PATH.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path
import mediapipe as mp
import json
from datetime import datetime, timezone

validation_results = []

for record in batch_records:
    image_path = Path(record["path"]) if record.get("path") else None
    if record["status"] != "GENERATED" or image_path is None or not image_path.exists():
        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": 0,
            "landmarks": 0,
            "valid": False,
            "reason": "generation_failed_or_file_missing",
        })
        continue

    try:
        mp_image = mp.Image.create_from_file(str(image_path))
        detection_result = landmarker.detect(mp_image)
        num_faces = len(detection_result.face_landmarks)
        num_landmarks = len(detection_result.face_landmarks[0]) if num_faces else 0
        valid = num_faces == 1 and num_landmarks == 478

        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": num_faces,
            "landmarks": num_landmarks,
            "valid": valid,
            "reason": None if valid else "expected_exactly_one_face_with_478_landmarks",
        })

        print(
            f"{record['filename']}: faces={num_faces}, "
            f"landmarks={num_landmarks}, valid={valid}"
        )
    except Exception as exc:
        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": 0,
            "landmarks": 0,
            "valid": False,
            "reason": f"validation_error: {type(exc).__name__}: {exc}",
        })
        print(f"{record['filename']}: validation error: {exc}")

validation_path = BATCH_DIR / "validation.json"
validation_document = {
    "batch_id": batch_id,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "checked_count": len(validation_results),
    "valid_count": sum(r["valid"] for r in validation_results),
    "invalid_count": sum(not r["valid"] for r in validation_results),
    "results": validation_results,
}
validation_path.write_text(json.dumps(validation_document, indent=2), encoding="utf-8")
print("Validation report:", validation_path)


In [ ]:
from pathlib import Path
import mediapipe as mp
import json
from datetime import datetime, timezone

validation_results = []

for record in batch_records:
    image_path = Path(record["path"]) if record.get("path") else None
    if record["status"] != "GENERATED" or image_path is None or not image_path.exists():
        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": 0,
            "landmarks": 0,
            "valid": False,
            "reason": "generation_failed_or_file_missing",
        })
        continue

    try:
        mp_image = mp.Image.create_from_file(str(image_path))
        detection_result = landmarker.detect(mp_image)
        num_faces = len(detection_result.face_landmarks)
        num_landmarks = len(detection_result.face_landmarks[0]) if num_faces else 0
        valid = num_faces == 1 and num_landmarks == 478

        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": num_faces,
            "landmarks": num_landmarks,
            "valid": valid,
            "reason": None if valid else "expected_exactly_one_face_with_478_landmarks",
        })

        print(
            f"{record['filename']}: faces={num_faces}, "
            f"landmarks={num_landmarks}, valid={valid}"
        )
    except Exception as exc:
        validation_results.append({
            "index": record["index"],
            "filename": record["filename"],
            "faces": 0,
            "landmarks": 0,
            "valid": False,
            "reason": f"validation_error: {type(exc).__name__}: {exc}",
        })
        print(f"{record['filename']}: validation error: {exc}")

validation_path = BATCH_DIR / "validation.json"
validation_document = {
    "batch_id": batch_id,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "checked_count": len(validation_results),
    "valid_count": sum(r["valid"] for r in validation_results),
    "invalid_count": sum(not r["valid"] for r in validation_results),
    "results": validation_results,
}
validation_path.write_text(json.dumps(validation_document, indent=2), encoding="utf-8")
print("Validation report:", validation_path)


In [ ]:
from pathlib import Path
import shutil

P01_DIR = Path("/kaggle/working/rhinoplasty_dataset/patients/P01")

candidate = P01_DIR / "normal" / "candidates" / "P01_candidate_0.png"
canonical = P01_DIR / "normal" / "canonical_front.png"

shutil.copy2(candidate, canonical)

print("✓ P01 canonical identity frozen")
print("Source :", candidate)
print("Frozen :", canonical)

In [ ]:
import inspect

print(inspect.signature(pipe.__call__))

In [ ]:
from PIL import Image
import torch
from pathlib import Path

P01_DIR = Path("/kaggle/working/rhinoplasty_dataset/patients/P01")

canonical_path = P01_DIR / "normal" / "canonical_front.png"

VIEW_DIR = P01_DIR / "normal" / "views"
VIEW_DIR.mkdir(parents=True, exist_ok=True)

canonical = Image.open(canonical_path).convert("RGB")

LEFT20_PROMPT = """
Using the provided reference image as the exact identity reference,
generate the same fictional person from a slightly turned
left-facing viewpoint.

Rotate the head approximately 20 degrees to the person's left.
Keep the same identity, facial proportions, age, skin appearance,
hair, hairstyle, facial features, and natural nose anatomy.

The person should maintain a neutral expression and look natural.
Do not change the person's identity.

The nose must remain the same normal nose as in the reference.
Do not create a hump, deviation, rhinoplasty change, or other
nasal deformation.

Photorealistic clinical-style portrait photography,
natural realistic skin texture, neutral studio lighting,
plain neutral background, single face.
"""

generator = torch.Generator(device="cpu").manual_seed(100)

print("Generating P01 Left20...")

with torch.inference_mode():
    result = pipe(
        image=canonical,
        prompt=LEFT20_PROMPT,
        height=384,
        width=384,
        num_inference_steps=16,
        guidance_scale=1.0,
        generator=generator,
    )

left20 = result.images[0]

left20_path = VIEW_DIR / "P01_left20.png"
left20.save(left20_path)

print("✓ Left20 generated")
print("Saved:", left20_path)
print("Size:", left20.size)


In [ ]:
image_path = "/kaggle/working/rhinoplasty_dataset/patients/P01/normal/views/P01_left20.png"

mp_image = mp.Image.create_from_file(image_path)
result = landmarker.detect(mp_image)

print("Faces:", len(result.face_landmarks))
print("Landmarks:", len(result.face_landmarks[0]) if result.face_landmarks else 0)
print(
    "Transformation matrices:",
    len(result.facial_transformation_matrixes)
)

In [ ]:
import numpy as np

matrix = np.array(
    result.facial_transformation_matrixes[0],
    dtype=np.float64
)

print("Matrix shape:", matrix.shape)

# Extract rotation component
R = matrix[:3, :3]

# Orthonormalize using SVD
U, _, Vt = np.linalg.svd(R)
R_ortho = U @ Vt

# Ensure proper rotation matrix
if np.linalg.det(R_ortho) < 0:
    U[:, -1] *= -1
    R_ortho = U @ Vt

# Euler-angle extraction
sy = np.sqrt(
    R_ortho[0, 0] ** 2 +
    R_ortho[1, 0] ** 2
)

singular = sy < 1e-6

if not singular:
    x = np.arctan2(R_ortho[2, 1], R_ortho[2, 2])
    y = np.arctan2(-R_ortho[2, 0], sy)
    z = np.arctan2(R_ortho[1, 0], R_ortho[0, 0])
else:
    x = np.arctan2(-R_ortho[1, 2], R_ortho[1, 1])
    y = np.arctan2(-R_ortho[2, 0], sy)
    z = 0

pitch = np.degrees(x)
yaw = np.degrees(y)
roll = np.degrees(z)

print("\n===== P01 LEFT20 =====")
print(f"Target yaw   : -20.00°")
print(f"Measured yaw : {yaw:.2f}°")
print(f"Pitch        : {pitch:.2f}°")
print(f"Roll         : {roll:.2f}°")
print(f"Yaw error    : {abs(-20 - yaw):.2f}°")

In [ ]:
from pathlib import Path
import shutil

P01_DIR = Path("/kaggle/working/rhinoplasty_dataset/patients/P01")

source = P01_DIR / "normal" / "views" / "P01_left20.png"
frozen_dir = P01_DIR / "normal" / "frozen_views"
frozen_dir.mkdir(parents=True, exist_ok=True)

destination = frozen_dir / "P01_left20.png"

shutil.copy2(source, destination)

print("✓ P01 Left20 frozen")
print("Saved:", destination)

In [ ]:
from IPython.display import display
from PIL import Image

front = Image.open(
    "/kaggle/working/rhinoplasty_dataset/patients/P01/normal/canonical_front.png"
)

left20 = Image.open(
    "/kaggle/working/rhinoplasty_dataset/patients/P01/normal/views/P01_left20.png"
)

display(front)
display(left20)

In [ ]:
from PIL import Image
import torch

canonical = Image.open(
    "/kaggle/working/rhinoplasty_dataset/patients/P01/normal/canonical_front.png"
).convert("RGB")

LEFT40_PROMPT = """
Using the provided reference image as the exact identity reference,
generate the same fictional person from a moderately turned
left-facing viewpoint.

Rotate the head approximately 40 degrees to the person's left.

Keep the same identity, facial proportions, age, skin appearance,
hair, hairstyle, facial features, and natural nose anatomy.

The person should maintain a neutral expression and look natural.
Do not change the person's identity.

The nose must remain the same normal nose as in the reference.
Do not create a hump, deviation, rhinoplasty change, or other
nasal deformation.

Photorealistic clinical-style portrait photography,
natural realistic skin texture, neutral studio lighting,
plain neutral background, single face.
"""

generator = torch.Generator(device="cpu").manual_seed(101)

with torch.inference_mode():
    result = pipe(
        image=canonical,
        prompt=LEFT40_PROMPT,
        height=384,
        width=384,
        num_inference_steps=16,
        guidance_scale=1.0,
        generator=generator,
    )

left40 = result.images[0]

left40_path = VIEW_DIR / "P01_left40.png"
left40.save(left40_path)

print("Saved:", left40_path)

In [ ]:
from IPython.display import display

display(left40)

In [ ]:
import numpy as np
from PIL import Image
import mediapipe as mp

left40_img = Image.open(
    "/kaggle/working/rhinoplasty_dataset/patients/P01/normal/views/P01_left40.png"
).convert("RGB")

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=np.array(left40_img)
)

result = landmarker.detect(mp_image)

print("Faces:", len(result.face_landmarks))
print("Landmarks:", len(result.face_landmarks[0]) if result.face_landmarks else 0)
print("Transformation matrices:",
      len(result.facial_transformation_matrixes)
      if result.facial_transformation_matrixes else 0)

In [ ]:
matrix = np.array(result.facial_transformation_matrixes[0])

R = matrix[:3, :3]

# Orthonormalize rotation matrix
U, _, Vt = np.linalg.svd(R)
R_ortho = U @ Vt

if np.linalg.det(R_ortho) < 0:
    U[:, -1] *= -1
    R_ortho = U @ Vt

sy = np.sqrt(
    R_ortho[0, 0]**2 +
    R_ortho[1, 0]**2
)

singular = sy < 1e-6

if not singular:
    x = np.arctan2(R_ortho[2, 1], R_ortho[2, 2])
    y = np.arctan2(-R_ortho[2, 0], sy)
    z = np.arctan2(R_ortho[1, 0], R_ortho[0, 0])
else:
    x = np.arctan2(-R_ortho[1, 2], R_ortho[1, 1])
    y = np.arctan2(-R_ortho[2, 0], sy)
    z = 0

pitch = np.degrees(x)
yaw = np.degrees(y)
roll = np.degrees(z)

target_yaw = -40.0
yaw_error = abs(yaw - target_yaw)

print(f"Target yaw   : {target_yaw:.2f}°")
print(f"Measured yaw : {yaw:.2f}°")
print(f"Yaw error    : {yaw_error:.2f}°")
print(f"Pitch        : {pitch:.2f}°")
print(f"Roll         : {roll:.2f}°")